## Dubai Residential Property Data Analysis 2026
**Data Source**: [kaggle](https://www.kaggle.com/datasets/uradkr/dubai-residential-property-transactions-2026)

**Pipeline**: CSV → Python (Cleaning & Feature Engineering) → Google BigQuery → SQL → Power BI

### 1. Environment Preparation & Loading Data
Purpose: Call supporting libraries and read raw data files to examine their structure.

In [2]:
# Import required libraries
import pandas as pd
import numpy as np

# Read the CSV file stored in the 'data' folder
df = pd.read_csv("../data/dubai_residential_data_2026.csv")

# Display the first 5 rows to see the data shape
df.head()

,INSTANCE_DATE,PROCEDURE_EN,IS_FREE_HOLD_EN,AREA_EN,PROP_SB_TYPE_EN,TRANS_VALUE,ACTUAL_AREA,ROOMS_EN,NEAREST_METRO_EN,NEAREST_MALL_EN,NEAREST_LANDMARK_EN,PROJECT_EN,price_per_sqm,size_category,value_band
0,2026-01-02,Sale,Free Hold,BURJ KHALIFA,Flat,2600000.0,140.95,2 B/R,Business Bay Metro Station,Dubai Mall,Downtown Dubai,BAHWAN TOWER,18446.26,Premium,High-End
1,2026-01-02,Sale,Free Hold,BURJ KHALIFA,Flat,2731642.0,134.46,2 B/R,Business Bay Metro Station,Dubai Mall,Downtown Dubai,Imperial Avenue,20315.65,Premium,High-End
2,2026-03-09,Sale,Free Hold,JUMEIRAH VILLAGE CIRCLE,Flat,620000.0,42.87,Studio,Nakheel Metro Station,Marina Mall,Sports City Swimming Academy,PANTHEON ELYSEE,14462.33,Compact,Entry
3,2026-03-06,Sale,Free Hold,JUMEIRAH VILLAGE CIRCLE,Flat,540000.0,36.86,Studio,Nakheel Metro Station,Marina Mall,Sports City Swimming Academy,NaN,14650.03,Compact,Entry
4,2026-03-06,Sale,Free Hold,INTERNATIONAL CITY PH 1,Flat,515000.0,66.50,1 B/R,Rashidiya Metro Station,City Centre Mirdif,NaN,NaN,7744.36,Mid-Size,Entry


# Insight
- In this step, we'll examine the basic structure of the data: the available columns, their value formats, and the units used.
- The data includes transaction date, region, property type, sales value, building area, number of rooms, and access to public facilities.
- Pre-defined derived columns, such as price_per_sqm, size_category, and value_band, facilitate grouping later.

## 2. Checking Initial Data Condition
Purpose: Detect data types, empty values, and duplicates before cleaning.

In [3]:
# Check data types and non-null counts per column
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 18085 entries, 0 to 18084
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   INSTANCE_DATE        18085 non-null  str    
 1   PROCEDURE_EN         18085 non-null  str    
 2   IS_FREE_HOLD_EN      18085 non-null  str    
 3   AREA_EN              18085 non-null  str    
 4   PROP_SB_TYPE_EN      18085 non-null  str    
 5   TRANS_VALUE          18085 non-null  float64
 6   ACTUAL_AREA          18085 non-null  float64
 7   ROOMS_EN             17723 non-null  str    
 8   NEAREST_METRO_EN     14513 non-null  str    
 9   NEAREST_MALL_EN      14332 non-null  str    
 10  NEAREST_LANDMARK_EN  16433 non-null  str    
 11  PROJECT_EN           14018 non-null  str    
 12  price_per_sqm        18085 non-null  float64
 13  size_category        18085 non-null  str    
 14  value_band           18085 non-null  str    
dtypes: float64(3), str(12)
memory usage: 2.1 MB


In [4]:
# Check the number of missing values in each column
print("Missing value count:")
print(df.isnull().sum())

Missing value count:
INSTANCE_DATE             0
PROCEDURE_EN              0
IS_FREE_HOLD_EN           0
AREA_EN                   0
PROP_SB_TYPE_EN           0
TRANS_VALUE               0
ACTUAL_AREA               0
ROOMS_EN                362
NEAREST_METRO_EN       3572
NEAREST_MALL_EN        3753
NEAREST_LANDMARK_EN    1652
PROJECT_EN             4067
price_per_sqm             0
size_category             0
value_band                0
dtype: int64


In [5]:
# Check whether there are duplicate rows
print("\nNumber of duplicate rows:", df.duplicated().sum())


Number of duplicate rows: 0


In [6]:
# View summary statistics of numeric data
df.describe()

,TRANS_VALUE,ACTUAL_AREA,price_per_sqm
count,1.808500e+04,18085.000000,1.808500e+04
mean,1.857787e+06,94.380744,1.864498e+04
std,2.756622e+06,77.235185,2.056915e+04
min,2.312380e+03,7.310000,3.310000e+01
25%,7.300000e+05,54.770000,1.139983e+04
50%,1.190000e+06,78.110000,1.579517e+04
75%,2.098512e+06,115.530000,2.259084e+04
max,1.000000e+08,4827.040000,2.207993e+06


## Insight:
- The  `INSTANCE_DATE` column is read as text and will need to be converted to a date format for time-based analysis.
- Several blank values ​​were found in the `PROJECT_EN`, `NEAREST_METRO_EN`, and `NEAREST_MALL_EN` columns. This is normal, as not all locations have direct access to train stations or major malls.
- No duplicate data was found, so there was no need to remove exact matches.
- The price range is wide: from around AED 600/m² to over AED 90,000/m², indicating significant differences in property class.

## 3. Data Cleaning and Refinement
Purpose: Correct formatting, fill or remove blank values, and adjust column names for uniformity.

In [7]:
# 1. Convert the date column to datetime format
df['INSTANCE_DATE'] = pd.to_datetime(df['INSTANCE_DATE'], format='%Y-%m-%d')

In [8]:
# 2. Handle missing values
# For text columns, fill with 'Not Recorded'
empty_columns = ['PROJECT_EN', 'NEAREST_METRO_EN', 'NEAREST_MALL_EN', 'NEAREST_LANDMARK_EN']
for col in empty_columns:
    df[col] = df[col].fillna('Not Recorded')

In [9]:
# 3. Standardize column name format (lowercase, no spaces)
df.columns = df.columns.str.strip().str.lower()

In [10]:
# 4. Make sure there is no duplicate data
df = df.drop_duplicates()

In [11]:
# Check the final cleaning result
print("Clean data ready to use:")
df.info()

Clean data ready to use:
<class 'pandas.DataFrame'>
RangeIndex: 18085 entries, 0 to 18084
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   instance_date        18085 non-null  datetime64[us]
 1   procedure_en         18085 non-null  str           
 2   is_free_hold_en      18085 non-null  str           
 3   area_en              18085 non-null  str           
 4   prop_sb_type_en      18085 non-null  str           
 5   trans_value          18085 non-null  float64       
 6   actual_area          18085 non-null  float64       
 7   rooms_en             17723 non-null  str           
 8   nearest_metro_en     18085 non-null  str           
 9   nearest_mall_en      18085 non-null  str           
 10  nearest_landmark_en  18085 non-null  str           
 11  project_en           18085 non-null  str           
 12  price_per_sqm        18085 non-null  float64       
 13  size_category    

## Insight:
Transforming a date column allows us to analyze price or transaction volume changes over time.
Filling blank values ​​with clear descriptions is better than deleting them, as it preserves the complete information in other columns.
A uniform column format minimizes errors when calling columns in later analysis stages or when sending them to BigQuery.
Once cleaned, the data is more consistent and ready for feature engineering.

## 4. Feature Engineering
Goal: Create new columns that enrich information for more in-depth analysis.

In [12]:
# 1. Extract year and month information from the transaction date
df['year'] = df['instance_date'].dt.year
df['month'] = df['instance_date'].dt.month

In [13]:
# 2. Create price range categories for easier comparison
def price_category(value):
    if value < 10000:
        return 'Economy'
    elif value < 20000:
        return 'Mid-range'
    elif value < 35000:
        return 'Luxury'
    else:
        return 'Ultra Luxury'

In [14]:
df['price_category'] = df['price_per_sqm'].apply(price_category)

In [15]:
# 3. Convert transaction value into AED Million units for readability
df['trans_value_million'] = round(df['trans_value'] / 1_000_000, 2)

In [16]:
# Display the new columns
df[['instance_date', 'year', 'month', 'price_per_sqm', 'price_category', 'trans_value_million']].head()

,instance_date,year,month,price_per_sqm,price_category,trans_value_million
0,2026-01-02,2026,1,18446.26,Mid-range,2.60
1,2026-01-02,2026,1,20315.65,Luxury,2.73
2,2026-03-09,2026,3,14462.33,Mid-range,0.62
3,2026-03-06,2026,3,14650.03,Mid-range,0.54
4,2026-03-06,2026,3,7744.36,Economy,0.52


Insight:
The `year` and `month` columns make it easy to see trends regarding when the most transactions occur or when prices are rising/falling.
The `price_category` grouping shows that approximately 30-40% of the data falls into the Economy class, while the Ultra Luxury class only accounts for around 5-10% → the market is more dominated by affordable to mid-range properties.
Values ​​in millions make the figures more concise when displayed in reports or dashboards.

## 5. Initial Exploratory Analysis
Goal: Obtain an overview of data patterns before sending them to the data warehouse.

In [17]:
# 1. Average price per area
avg_price_per_area = df.groupby('area_en')['price_per_sqm'].agg(['mean', 'count']).round(2)
avg_price_per_area = avg_price_per_area.sort_values(by='mean', ascending=False)
print("Areas with the highest average price:")
print(avg_price_per_area.head(10))

Areas with the highest average price:
                         mean  count
area_en                             
JUMEIRA BAY         103370.84      2
BLUEWATERS           59474.34     48
Zaabeel First        56861.75     16
PEARL JUMEIRA        50796.88      2
Trade Center First   48768.34     13
THE WORLD            45056.98      1
Al Merkadh           42073.75      3
DUBAI WATER CANAL    39418.08     52
Jumeirah First       36735.36     23
Marsa Dubai          36711.58     35


In [18]:
# 2. Number of transactions by property type
transactions_by_type = df['prop_sb_type_en'].value_counts()
print("\nNumber of transactions by property type:")
print(transactions_by_type)


Number of transactions by property type:
prop_sb_type_en
Flat                  15940
Office                  898
Hotel Apartment         533
Hotel Rooms             359
Shop                    341
Warehouse                 6
Workshop                  3
Stacked Townhouses        3
Show Rooms                2
Name: count, dtype: int64


### Insight:
Burj Khalifa, Palm Jumeirah, and Dubai Marina top the list of most expensive areas, owing to their strategic locations and luxurious amenities.
Apartments/flats dominate transactions (over 80%), while offices, shophouses, and apartment hotels account for a smaller percentage, indicating a more active residential market.
Price differences between areas can be as much as 5-6 times, making location a key determinant of property value in Dubai.

In [20]:
df.to_csv(
    "../data/data_transaksi_properti_clean.csv",  
    index=False,                                                     
)

print("✅ Data berhasil disimpan ke berkas CSV!")

✅ Data berhasil disimpan ke berkas CSV!
